# 🍔 Lab 2A — Why Is AI So Hungry?
**Foothold · GPU Computing for AI · Unit 2 (first half) · Book: Step 2, Sections 2.1–2.4**

Unit 1 showed that AI is multiply-adds. This lab **measures** how many it does and how fast an ordinary CPU can do them, and it finds the work that can be done in parallel.

| Part | What you do | Goes with Foothold lesson |
|---|---|---|
| A | Count the parameters and FLOPs of real models | 2.1 Counting the Multiply-Adds |
| B | Time inference vs training on the CPU | 2.2 Training vs Inference |
| C | Loops vs vectorised code: independent work vs a chain | 2.3 Every Output Is Independent |
| D | Measure your CPU's real speed in GFLOPS, then run the compute bill | 2.4 Capstone: The Compute Bill |

⏱️ About 90 minutes. 📘 Textbook: Hamacher Ch 1 (Performance) · Stallings §2.4 · D2L Ch 12–13. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`) saying what it does. Read them as you go.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **No GPU needed for this lab.** The default CPU runtime is fine — that is deliberate: you will compare against the GPU later.
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
# time: a stopwatch, time.perf_counter()
import time
# NumPy: arrays of numbers and fast maths on them
import numpy as np
# PyTorch, the AI library, and nn, its building blocks (layers, losses)
import torch
from torch import nn   # nn: PyTorch's building blocks (layers, losses)

def fmt(x):   # a helper that writes a big number in words
    # turn a huge number into words, e.g. 3.6e9 becomes "3.60 billion"
    for unit, size in [("quadrillion", 1e15), ("trillion", 1e12), ("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]:
        # use the biggest unit the number reaches
        if x >= size:
            return f"{x / size:.2f} {unit}"   # e.g. 3.6e9 → "3.60 billion"
    # small numbers are printed as they are
    return f"{x:.0f}"

print("Ready. PyTorch", torch.__version__.split("+")[0])   # show that everything loaded, and PyTorch's version

---
## Part A · Count the Work  *(Unit 2.1)*

**Parameters** are the knobs (weights + biases). For a fully connected (`Linear`) layer with `n_in` inputs and `n_out` neurons:
- parameters = `n_in × n_out + n_out`
- multiply-adds (MACs) per input = `n_in × n_out`  →  FLOPs = 2 × MACs

Then a real photo model, **ResNet-18**, built with random weights (`weights=None`), so nothing is downloaded: we only want to count it.

In [ ]:
# the 784 → 128 → 10 digit network from Lab 1
lab1_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
# count every weight and bias: .numel() is how many numbers are in each tensor
params = sum(p.numel() for p in lab1_model.parameters())
# layer 1: 784 inputs × 128 neurons; layer 2: 128 inputs × 10 neurons
macs = 784 * 128 + 128 * 10
print(f"Lab 1 network: {params:,} parameters, {macs:,} MACs = {2 * macs:,} FLOPs per image")   # one MAC = 2 FLOPs

# torchvision holds famous ready-made model designs
from torchvision import models
# ResNet-18, a real photo model; weights=None builds it with random weights, so nothing is downloaded
resnet = models.resnet18(weights=None)
resnet_params = sum(p.numel() for p in resnet.parameters())   # count all of ResNet-18's weights and biases
print(f"ResNet-18: {resnet_params:,} parameters (about {fmt(resnet_params)})")   # show the count, and in words
print("Published figure: about 1.8 billion MACs (3.6 billion FLOPs) per 224 × 224 photo")   # its work per photo, from its designers

**Chatbots:** writing one token costs about **2 FLOPs per parameter**.

In [ ]:
# three model sizes, in parameters: 1 billion, 7 billion and 175 billion
for name, p in [("Small open model", 1e9), ("7B model", 7e9), ("GPT-3 class", 175e9)]:
    # rule of thumb: 2 FLOPs per parameter for every token the model writes
    per_token = 2 * p
    # a 200-word answer is about 267 tokens (a token is about ¾ of a word)
    answer = per_token * 267
    print(f"{name:17s}: {fmt(per_token):>17s} FLOPs per token, {fmt(answer):>18s} FLOPs per 200-word answer")   # :17s and :>17s pad the text so the columns line up

### ✏️ YOUR TURN (A)
1. Change the Lab 1 network to `784 → 512 → 512 → 10`. Recompute the parameters and MACs **by hand** first, then check with code.
2. How many times more FLOPs does ResNet-18 need per image than your Lab 1 network?

*Your answers:*

---
## Part B · Inference vs Training  *(Unit 2.2)*

Inference = forward pass only. Training = forward + backward + update. Time 100 of each on the same batch.

In [ ]:
# fix the random numbers so the run can be repeated
torch.manual_seed(0)
# a bigger digit network: 784 → 512 → 512 → 10
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
# a batch of 256 made-up images in MNIST's shape, and 256 made-up labels (digits 0–9)
x = torch.randn(256, 1, 28, 28)
y = torch.randint(0, 10, (256,))   # 256 made-up labels, each a digit 0–9
# the loss for "which class is it?" problems, and plain gradient descent as the optimiser
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.01)   # plain gradient descent, learning rate 0.01

# inference: forward passes only; no_grad tells PyTorch not to keep anything for a backward pass
with torch.no_grad():
    t0 = time.perf_counter()   # start the stopwatch
    for _ in range(100):   # repeat 100 times ("_" = we don't need the counter)
        model(x)   # one forward pass over the whole batch
    t_infer = time.perf_counter() - t0   # stop: time for 100 inference passes

# training: forward pass + loss, clear old slopes, backward pass, update the weights
t0 = time.perf_counter()
for _ in range(100):   # repeat 100 times ("_" = we don't need the counter)
    loss = loss_fn(model(x), y)   # forward pass, then the loss
    opt.zero_grad()   # clear the old slopes
    loss.backward()   # backward pass: the slope of every weight
    opt.step()   # nudge every weight
t_train = time.perf_counter() - t0   # stop: time for 100 training steps

print(f"100 inference passes : {t_infer:.2f} s")   # show both times...
print(f"100 training steps   : {t_train:.2f} s")   # ...side by side
print(f"Training / inference : {t_train / t_infer:.1f}×   (theory says about 3×)")   # and how many times longer training took

The theory counts only multiply-adds (about 3×). On a CPU, a small network also spends time on everything *around* the arithmetic, so your ratio may well be higher.

**Epochs, batches, iterations.** For MNIST (60,000 images):

In [ ]:
# MNIST has 60,000 training images (the _ is just a digit separator, like a comma)
N = 60_000
for batch in [1, 32, 64, 256, 1024]:   # try five batch sizes
    # round UP: a last, part-full batch still counts as one iteration
    iters = -(-N // batch)
    print(f"batch {batch:5d} → {iters:6,d} iterations per epoch")   # show the iterations needed for one epoch

### ✏️ YOUR TURN (B)
1. Was your ratio close to 3×? Give one reason it might differ.
2. Use **training FLOPs ≈ 6 × parameters × examples** to estimate the FLOPs for 3 epochs of the Lab 1 network on MNIST. (Treat one image as one "token".)

*Your answers:*

---
## Part C · Independent Work vs a Chain  *(Unit 2.3)*

**ReLU** changes each number on its own, so it's independent work. A **running total** needs the previous total each time, so it's a chain.

In [ ]:
# 10 lakh random numbers, some negative and some positive
data = np.random.default_rng(0).standard_normal(1_000_000)

# ReLU one number at a time in plain Python: keep the positives, turn negatives into 0
t0 = time.perf_counter()
relu_loop = [v if v > 0 else 0.0 for v in data]   # ReLU one number at a time in Python
t_loop = time.perf_counter() - t0   # stop: the loop's time

# the same ReLU with NumPy, on the whole array at once
t0 = time.perf_counter()
relu_vec = np.maximum(data, 0)   # ReLU on the whole array at once with NumPy
t_vec = time.perf_counter() - t0   # stop: NumPy's time

# × 1000 turns seconds into milliseconds
print(f"ReLU, Python loop : {t_loop * 1000:7.1f} ms")
print(f"ReLU, NumPy       : {t_vec * 1000:7.2f} ms   ({t_loop / t_vec:.0f}× faster)")   # and how many times faster NumPy was

In [ ]:
# a running total: every step NEEDS the total from the step before
t0 = time.perf_counter()
running, total = [], 0.0   # the list of running totals, and the total so far
for v in data:   # one number at a time, in order
    # add this number to the total so far, and remember the new total
    total += v
    running.append(total)   # remember the total after this step
t_chain_loop = time.perf_counter() - t0   # stop: the loop's time

# np.cumsum (cumulative sum) is compiled code, but it still has to walk the chain in order
t0 = time.perf_counter()
running_np = np.cumsum(data)   # the same running total with NumPy
t_chain_np = time.perf_counter() - t0   # stop: NumPy's time

print(f"Running total, Python loop : {t_chain_loop * 1000:7.1f} ms")   # show both times
print(f"Running total, np.cumsum   : {t_chain_np * 1000:7.2f} ms")   # (× 1000 turns seconds into ms)
# allclose: the two answers are equal apart from tiny rounding
print("Same result:", np.allclose(running, running_np))

Both NumPy versions are faster than the loops, because they're compiled code with no Python overhead per step. But there's a big difference in what they could become:
- **ReLU** could be split across 1,000 workers, each doing 1,000 numbers at the same moment.
- The **running total** can't simply be split that way: each value waits for the previous one.

GPUs speed up the first kind enormously. That's Lab 2B.

### ✏️ YOUR TURN (C)
Which of these is independent work and which is a chain? (a) adding a bias to every neuron, (b) computing compound interest year by year, (c) converting 10,000 photos to greyscale, (d) a chatbot writing its answer word by word.

*Your answers:*

---
## Part D · Measure Your CPU's Speed  *(Unit 2.4)*

An n × n matrix multiply needs **n³ MACs = 2n³ FLOPs**. Time it, divide, and you have your machine's real FLOPS.

In [ ]:
def py_matmul(A, B):   # a matrix multiply written in plain Python
    # matrix multiply in plain Python: C[i][j] = the sum over k of A[i][k] × B[k][j]
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]   # every cell C[i][j] = sum over k of A[i][k] × B[k][j]

# small sizes only: plain Python is slow
py_sizes, py_times = [20, 40, 60, 80], []
for n in py_sizes:   # for each size
    # two random n × n matrices, as plain Python lists
    A = np.random.rand(n, n).tolist(); B = np.random.rand(n, n).tolist()
    # time one multiply and keep the time for the chart
    t0 = time.perf_counter(); py_matmul(A, B); t = time.perf_counter() - t0
    py_times.append(t)   # keep the time for later
    # an n × n multiply is 2n³ FLOPs; FLOPs ÷ seconds ÷ 1e6 = MFLOPS (millions per second)
    print(f"Python  n={n:4d}: {t * 1000:9.1f} ms   → {2 * n**3 / t / 1e6:8.1f} MFLOPS")

In [ ]:
# NumPy is fast enough for much bigger matrices
np_sizes, np_times = [100, 200, 400, 800, 1600], []
for n in np_sizes:   # for each size
    # float32 = 4-byte numbers, the usual size in AI
    A = np.random.rand(n, n).astype(np.float32); B = np.random.rand(n, n).astype(np.float32)
    # a warm-up run first (the first call has extra start-up cost), then the timed run
    A @ B
    t0 = time.perf_counter(); A @ B; t = time.perf_counter() - t0   # time one multiply (@ is NumPy's matrix multiply)
    np_times.append(t)   # keep the time for later
    # ÷ 1e9 gives GFLOPS (billions per second)
    print(f"NumPy   n={n:4d}: {t * 1000:9.2f} ms   → {2 * n**3 / t / 1e9:8.1f} GFLOPS")

# the best speed over all sizes is your CPU's measured speed (used in the compute bill)
cpu_flops = max(2 * n**3 / t for n, t in zip(np_sizes, np_times))
print(f"\nYour CPU's best measured speed: {cpu_flops / 1e9:.1f} GFLOPS")   # the backslash-n at the start prints an empty line first

In [ ]:
import matplotlib.pyplot as plt                                # Matplotlib: charts
plt.figure(figsize=(6, 4))                                     # a new 6 × 4 inch chart
plt.loglog(py_sizes, py_times, "o-", label="plain Python")     # log scale on both axes; circles joined by lines
plt.loglog(np_sizes, np_times, "s-", label="NumPy")            # squares joined by lines
plt.xlabel("matrix size n"); plt.ylabel("time (s)")            # axis labels
plt.grid(alpha=0.3, which="both"); plt.legend()                # faint grid and the colour key
plt.title("Doubling n → about 8× the time (n³)")               # the chart's title
plt.show()                                                     # draw it

### The compute bill, with YOUR measured CPU
This is the capstone estimator, with the guessed laptop speed replaced by the number you just measured.

In [ ]:
# the helpdesk: a 7-billion-parameter model, 5,000 students, 4 questions each per day, 200-word answers
params, students, questions_each, words = 7e9, 5000, 4, 200
# assumptions you can change: real programs reach about 30% of peak; ₹35 per GPU-hour
utilisation, rupees_per_hour = 0.3, 35

# work per day = 2 FLOPs per parameter per token × tokens per answer (words ÷ 0.75) × answers per day
flops_per_day = 2 * params * (words / 0.75) * students * questions_each
print(f"Work per day: {flops_per_day:.2e} FLOPs\n")   # .2e writes it like 7.47e+16

# each machine's peak speed in FLOPS; your CPU is the value you just measured
machines = {"YOUR CPU (measured)": cpu_flops, "T4 GPU FP32": 8.1e12, "T4 GPU FP16": 65e12}
for name, peak in machines.items():   # for each machine and its peak speed
    # time = work ÷ real speed; ÷ 3600 turns seconds into hours
    hours = flops_per_day / (peak * utilisation) / 3600
    # more than 24 hours of work per day means one machine can't keep up: how many do we need?
    verdict = "fits in one day" if hours <= 24 else f"needs {int(hours // 24) + 1} machines"
    print(f"{name:20s}: {hours:10.1f} hours of work per day   {verdict}")   # show its hours per day and the verdict

### ✏️ YOUR TURN (D) — record your results
| | Your value |
|---|---|
| Plain Python speed (MFLOPS) | |
| NumPy best speed (GFLOPS) | |
| NumPy ÷ Python | |
| Hours/day for the bot on your CPU | |

**Think:** NumPy is hundreds or thousands of times faster than plain Python on the *same* CPU. Where could that speed come from? (Hint: compiled code, SIMD instructions, several cores, and cache-friendly order. Lab 2B continues the story.)

*Your answers:*

---
## 🎓 Wrap-up
- time = work (FLOPs) ÷ speed (FLOPS). You can now estimate any AI job.
- Training ≈ 3× inference per example, repeated over the whole dataset every epoch.
- Matrix multiplies and element-wise operations are **independent work**: perfect for many workers at once.
- **Next — Lab 2B: CPU vs GPU.** You'll run your first code on a GPU and race it against the CPU numbers you recorded today.